# Sarvam-m 24B — Remote Ollama Endpoint

Minimal setup: GPU → Ollama → Sarvam-m → ngrok public URL.


In [11]:
# 1. Verify GPU
!nvidia-smi


Mon Aug 31 20:36:49 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.159.04             Driver Version: 580.159.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   43C    P8             12W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [12]:
# 2. Install Ollama
!apt-get update -qq
!apt-get install -y -qq zstd
!curl -fsSL https://ollama.com/install.sh | sh


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%###################                       71.8%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [22]:
# 3. Start Ollama and wait until the API is ready
import subprocess
import time
import requests

subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

for _ in range(30):
    try:
        r = requests.get("http://127.0.0.1:11434/api/tags", timeout=2)
        if r.status_code == 200:
            print("Ollama server is ready")
            break
    except requests.RequestException:
        pass
    time.sleep(1)
else:
    raise RuntimeError("Ollama server did not start")


Ollama server is ready


In [23]:
# 4. Restore Sarvam-m from Drive cache, or pull fresh (Colab auth + Drive API)
MODEL = "hf.co/tifin-india/sarvam-m-24b-q4-k-m-gguf:latest"
OLLAMA_MODELS = os.path.expanduser("~/.ollama/models")

import os, glob, subprocess, io

def drive_file_id():
    try:
        subprocess.run(["pip", "install", "-q", "google-auth", "google-auth-httplib2", "google-api-python-client"], check=False)
        from google.colab import auth
        from googleapiclient.discovery import build
        auth.authenticate_user()
        s = build('drive', 'v3')
        folders = s.files().list(q="mimeType='application/vnd.google-apps.folder' and trashed=false",
                                 fields="files(id,name)").execute().get('files', [])
        fid = next((f['id'] for f in folders if f['name'] == 'sarvam-ollama'), None)
        if fid is None:
            return None, None
        files = s.files().list(q=f"'{fid}' in parents and trashed=false",
                               fields="files(id,name,size)").execute().get('files', [])
        for f in files:
            if f['name'].startswith('sha256-') and int(f.get('size', 0)) > 1e9:
                return s, f
        return None, None
    except Exception as e:
        print("Drive lookup skipped/failed:", type(e).__name__, str(e)[:120])
        return None, None

drive_svc, remote = drive_file_id()
if remote is not None:
    os.makedirs(f"{OLLAMA_MODELS}/blobs", exist_ok=True)
    dst = f"{OLLAMA_MODELS}/blobs/{remote['name']}"
    if os.path.exists(dst):
        print(f"Model already restored locally ({os.path.getsize(dst)/1e9:.1f} GB) - skipping.")
    else:
        print(f"Restoring {remote['name']} from Drive (~14 GB download)...")
        from googleapiclient.http import MediaIoBaseDownload
        req = drive_svc.files().get_media(fileId=remote['id'])
        fh = io.BytesIO()
        dl = MediaIoBaseDownload(fh, req)
        done = False
        while not done:
            status, done = dl.next_chunk()
            if status:
                print(f"  {int(status.progress()*100)}%")
        fh.seek(0)
        with open(dst, 'wb') as out:
            out.write(fh.read())
    print("Restored from Drive cache.")
else:
    print("No Drive cache (or auth skipped) - pulling Sarvam-m (~10-20 min)...")
    r = subprocess.run(["ollama", "pull", MODEL],
                       stdout=subprocess.DEVNULL, stderr=subprocess.PIPE, text=True)
    if r.returncode != 0:
        print(r.stderr)
        raise RuntimeError("Sarvam-m download failed")
    print("Sarvam-m pulled")

!ollama list

No Drive cache (or auth skipped) - pulling Sarvam-m (~10-20 min)...
Sarvam-m pulled
]11;?\NAME                                                 ID              SIZE     MODIFIED      
hf.co/tifin-india/sarvam-m-24b-q4-k-m-gguf:latest    d1a77573ede7    14 GB    5 seconds ago    


In [26]:
# 4a. Cache Sarvam-m weights to Drive (Colab auth + Drive API) - run ONCE
import os, glob, subprocess
subprocess.run(["pip", "install", "-q", "google-auth", "google-auth-httplib2", "google-api-python-client"], check=False)

from google.colab import auth
from googleapiclient.discovery import build
from googleapiclient.http import MediaFileUpload

auth.authenticate_user()          # Google sign-in popup (no client_secrets.json needed)
drive_svc = build('drive', 'v3')

# find the ~14GB weights blob
OLLAMA_MODELS = os.path.expanduser("~/.ollama/models")
blobs = glob.glob(f"{OLLAMA_MODELS}/blobs/sha256-*")
big = max(blobs, key=os.path.getsize)
remote_name = os.path.basename(big)
print(f"Uploading: {remote_name} ({os.path.getsize(big)/1e9:.1f} GB)")

# find/create the sarvam-ollama folder
folder_id = None
res = drive_svc.files().list(q="mimeType='application/vnd.google-apps.folder' and trashed=false",
                             fields="files(id,name)").execute()
for f in res.get('files', []):
    if f['name'] == 'sarvam-ollama':
        folder_id = f['id']
        break
if folder_id is None:
    folder_id = drive_svc.files().create(body={
        'name': 'sarvam-ollama', 'mimeType': 'application/vnd.google-apps.folder'
    }, fields='id').execute()['id']

# skip if already uploaded
exists = drive_svc.files().list(
    q=f"'{folder_id}' in parents and name='{remote_name}' and trashed=false",
    fields="files(id)").execute().get('files')
if exists:
    print("Weights already on Drive - skipping upload.")
else:
    media = MediaFileUpload(big, mimetype='application/octet-stream', resumable=True)
    file_meta = {'name': remote_name, 'parents': [folder_id]}
    req = drive_svc.files().create(body=file_meta, media_body=media, fields='id')
    resp = None
    while resp is None:                       # resumable upload with progress
        status, resp = req.next_chunk()
        if status:
            print(f"  {int(status.progress()*100)}%")
    print("Uploaded weights to Drive. Done.")

Uploading: sha256-f79438888cfc607bee5b2b71f03517b674b07cd85eb26a26c43edea1d52e9335 (14.3 GB)
  0%
  1%
  2%
  2%
  3%
  4%
  5%
  5%
  6%
  7%
  8%
  8%
  9%
  10%
  10%
  11%
  12%
  13%
  13%
  14%
  15%
  16%
  16%
  17%
  18%
  19%
  19%
  20%
  21%
  21%
  22%
  23%
  24%
  24%
  25%
  26%
  27%
  27%
  28%
  29%
  29%
  30%
  31%
  32%
  32%
  33%
  34%
  35%
  35%
  36%
  37%
  38%
  38%
  39%
  40%
  40%
  41%
  42%
  43%
  43%
  44%
  45%
  46%
  46%
  47%
  48%
  49%
  49%
  50%
  51%
  51%
  52%
  53%
  54%
  54%
  55%
  56%
  57%
  57%
  58%
  59%
  59%
  60%
  61%
  62%
  62%
  63%
  64%
  65%
  65%
  66%
  67%
  68%
  68%
  69%
  70%
  70%
  71%
  72%
  73%
  73%
  74%
  75%
  76%
  76%
  77%
  78%
  79%
  79%
  80%
  81%
  81%
  82%
  83%
  84%
  84%
  85%
  86%
  87%
  87%
  88%
  89%
  89%
  90%
  91%
  92%
  92%
  93%
  94%
  95%
  95%
  96%
  97%
  98%
  98%
  99%
Uploaded weights to Drive. Done.


In [27]:
# 5. Create the public ngrok tunnel
!pip install -q pyngrok

from getpass import getpass
from pyngrok import ngrok

ngrok_token = getpass("Enter your ngrok authtoken: ")
ngrok.set_auth_token(ngrok_token)
ngrok.kill()

tunnel = ngrok.connect(
    addr=11434,
    proto="http",
    host_header="localhost:11434"
)

public_url = tunnel.public_url
print("PUBLIC URL:", public_url)


Enter your ngrok authtoken:  ········


PUBLIC URL: https://nonmutinously-oncological-meg.ngrok-free.dev


In [28]:
# 6. Verify that the public URL exposes Ollama
r = requests.get(f"{public_url}/api/tags", timeout=30)
print("Status:", r.status_code)
print(r.text)


Status: 200
{"models":[{"name":"hf.co/tifin-india/sarvam-m-24b-q4-k-m-gguf:latest","model":"hf.co/tifin-india/sarvam-m-24b-q4-k-m-gguf:latest","modified_at":"2026-08-31T20:45:55.719876363Z","size":14333911677,"digest":"d1a77573ede747bab2287a85c24b7e64a06fe80c8f0d2dc3f54e9c2ffacad738","details":{"parent_model":"","format":"gguf","family":"llama","families":["llama"],"parameter_size":"23.6B","quantization_level":"unknown","context_length":32768,"embedding_length":5120},"capabilities":["completion"]}]}


In [9]:
# 7. Optional: test one real Sarvam-m generation through the public URL
payload = {
    "model": MODEL,
    "prompt": "Answer entirely in Hindi, using Devanagari script: What is a good life goal related to self-image?",
    "stream": False,
    "think": False
}

r = requests.post(
    f"{public_url}/api/generate",
    json=payload,
    timeout=300
)

print("Status:", r.status_code)
print(r.text)


Status: 200
{"model":"hf.co/tifin-india/sarvam-m-24b-q4-k-m-gguf:latest","created_at":"2026-08-31T20:18:15.346494901Z","response":" एक अच्छा जीवन लक्ष्य जो आत्म-छवि से जुड़ा हो, वह है:  \n**\"स्वीकार्यता और आत्म-सम्मान को बढ़ाना।\"**  \n\nइसमें शामिल है:  \n1. **स्वीकार:** अपनी कमियों और गुणों को समझते हुए खुद को वैसा स्वीकार करना जैसा है।  \n2. **विकास:** नकारात्मक सोच या तुलना की आदतों को बदलकर सकारात्मक आत्म-छवि गढ़ना।  \n3. **स्वीकृति:** बाहरी मानकों के बजाय अपनी व्यक्तिगत मूल्यों और खुशी को प्राथमिकता देना।  \n\nएक स्वस्थ आत्म-छवि आत्मविश्वास, आंतरिक शांति और जीवन के अन्य पहलुओं (रिश्ते, कार्य, स्वास्थ्य) में संतुलन की नींव रखती है।","done":true,"done_reason":"stop","context":[17,4568,1584,1261,20351,27089,1046,18,3,31106,17262,1294,83301,1044,2505,2491,6596,1393,1859,8466,1058,5675,1395,1261,3683,4129,8406,6164,1317,2585,29328,1063,4,9765,4350,7013,36635,1658,4100,2063,42428,5006,2622,12152,4332,4100,2378,5569,2823,14789,1045,28865,4658,2069,2773,1803,4100,84710,7499,1658,2847,23

In [10]:
import time

time.sleep(2 * 60 * 60)

KeyboardInterrupt: 

In [29]:
# Cleanup when the pipeline is finished

from pyngrok import ngrok
import subprocess


# Stop the public tunnel
ngrok.kill()

# Stop Ollama
subprocess.run(["pkill", "-f", "ollama"], check=False)

print("ngrok and Ollama stopped.")

ngrok and Ollama stopped.
